### Import modules

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout
from tensorflow import keras
# Ensure reproducibility
np.random.seed(42)
tf.random.set_seed(42)


### Load Data

In [ ]:
data = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/pattern_embeddings/gemini_pattern_embedding_v3.csv')
# data = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/embeddings/Gemini_embeddings.csv')

data = data.drop(columns=['file'])
data['pattern'] = data['pattern'].fillna('None Type')
data.head()

### Configure

In [ ]:
EVALUATING_ENABLED = False
TEMP_TEST_SPLIT = False

In [ ]:
if TEMP_TEST_SPLIT:
    temp_train_data,temp_test_data = train_test_split(data, test_size=0.2, random_state=42, stratify=data['pattern'])
    data = temp_train_data
    temp_test_data.to_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/proba_distributions/temp_test_data/temp_test_data.csv', index=False)
    temp_train_data.to_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/proba_distributions/temp_test_data/temp_train_data.csv', index=False)

### NN Architecture

In [ ]:
TARGET_COLUMN = "pattern"

ARTIFACT_DIR = Path("../models/pattern_nn_classifier").resolve()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ARTIFACT_DIR / "pattern_classifier.keras"
SCALER_PATH = ARTIFACT_DIR / "scaler.joblib"
ENCODER_PATH = ARTIFACT_DIR / "label_encoder.joblib"
METADATA_PATH = ARTIFACT_DIR / "metadata.json"

In [ ]:
def build_classifier(input_dim: int, num_classes: int) -> Sequential:
    """Return a tuned dense network regularized for high-dimensional embeddings."""
    regularizer = tf.keras.regularizers.l2(1e-4)
    return Sequential(
        [
            tf.keras.Input(shape=(input_dim,)),
            Dense(768, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.35),
            Dense(512, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.3),
            Dense(256, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.25),
            Dense(128, activation="relu"),
            Dropout(0.2),
            # Dense(num_classes, activation="softmax"),
            Dense(num_classes, activation=None),
        ]
    )

@keras.utils.register_keras_serializable()
class TemperatureScaling(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()
        self.temperature = tf.Variable(
            initial_value=1.0,
            trainable=True,
            dtype=tf.float32,
            constraint=lambda t: tf.clip_by_value(t, 1e-6, 100.0),
        )

    def call(self, logits):
        return logits / self.temperature
    def get_config(self):
        return super().get_config()

def build_calibrated_model(base_model: tf.keras.Model) -> tf.keras.Model:
    base_model.trainable = False

    inputs = tf.keras.Input(shape=base_model.input_shape[1:])
    logits = base_model(inputs)

    scaled_logits = TemperatureScaling()(logits)
    outputs = tf.keras.layers.Softmax()(scaled_logits)

    return tf.keras.Model(inputs, outputs)

import numpy as np

def nn_train(data=data):
    numeric_features = data.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_features:
        raise ValueError("No numeric features detected – please ensure embeddings/features are present.")

    X = data[numeric_features].fillna(0.0).values
    y = data[TARGET_COLUMN].astype(str).values
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    num_classes = len(label_encoder.classes_)

    X_train, X_temp, y_train_enc, y_temp_enc = train_test_split(
        X,
        y_encoded,
        test_size=0.3,
        random_state=42,
        stratify=y_encoded,
    )
    X_val, X_test, y_val_enc, y_test_enc = train_test_split(
        X_temp,
        y_temp_enc,
        test_size=0.5,
        random_state=42,
        stratify=y_temp_enc,
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    y_train = tf.keras.utils.to_categorical(y_train_enc, num_classes)
    y_val = tf.keras.utils.to_categorical(y_val_enc, num_classes)
    y_test = tf.keras.utils.to_categorical(y_test_enc, num_classes)


    if not EVALUATING_ENABLED:
        # X_train = np.vstack([X_train, X_val])
        # y_train = np.vstack([y_train, y_val])
        X_val = np.vstack([X_val, X_test])
        y_val = np.vstack([y_val, y_test])

    def expected_calibration_error(
        probs: np.ndarray,
        y_true: np.ndarray,
        n_bins: int = 15,
    ) -> float:
        confidences = np.max(probs, axis=1)
        predictions = np.argmax(probs, axis=1)
        accuracies = (predictions == y_true).astype(float)

        bin_boundaries = np.linspace(0.0, 1.0, n_bins + 1)
        ece = 0.0
        N = len(y_true)

        for i in range(n_bins):
            bin_lower = bin_boundaries[i]
            bin_upper = bin_boundaries[i + 1]

            in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
            bin_size = np.sum(in_bin)

            if bin_size > 0:
                bin_accuracy = np.mean(accuracies[in_bin])
                bin_confidence = np.mean(confidences[in_bin])

                ece += (bin_size / N) * abs(bin_accuracy - bin_confidence)

        return ece


    model = build_classifier(X_train.shape[1], num_classes)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
        metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc")],
    )

    callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=20, min_delta=1e-4, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=8, min_lr=1e-5),
    ]

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=200,
        batch_size=64,
        callbacks=callbacks,
        verbose=1,
    )

    calibrated_model = build_calibrated_model(model)

    calibrated_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
        loss=tf.keras.losses.CategoricalCrossentropy(),
        metrics=[
            "accuracy",
            tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
        ],
    )


    calibrated_model.fit(
        X_val,
        y_val,
        epochs=50,
        batch_size=256,
        verbose=0,
    )


    if not EVALUATING_ENABLED:
        # Persist artifacts for downstream inference pipelines
        calibrated_model.save(MODEL_PATH, include_optimizer=True)
        joblib.dump(scaler, SCALER_PATH)
        joblib.dump(label_encoder, ENCODER_PATH)
        metadata = {
            "target_column": TARGET_COLUMN,
            "numeric_features": numeric_features,
            "num_classes": num_classes,
            "label_classes": label_encoder.classes_.tolist(),
        }
        METADATA_PATH.write_text(json.dumps(metadata, indent=2))
        print(f"Saved model to {MODEL_PATH}")
        print(f"Saved scaler to {SCALER_PATH}")
        print(f"Saved label encoder to {ENCODER_PATH}")
        print(f"Saved metadata to {METADATA_PATH}")
    else:
        print("\nEvaluation enabled; not saving model artifacts.")

        test_loss, test_acc, test_top3 = calibrated_model.evaluate(X_test, y_test, verbose=0)
        print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f} | Test top-3 accuracy: {test_top3:.4f}")

        y_pred = calibrated_model.predict(X_test)
        y_pred_labels = y_pred.argmax(axis=1)
        report = classification_report(
            y_test_enc,
            y_pred_labels,
            target_names=label_encoder.classes_,
            output_dict=True,
            zero_division=0,
        )
        report_df = pd.DataFrame(report).T
        summary = report_df.loc[["accuracy", "macro avg", "weighted avg"]].round(3)
        class_breakdown = (
            report_df.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("support", ascending=False).head(8).round(3)
        )
        probs = calibrated_model.predict(X_test)
        ece = expected_calibration_error(probs, y_test_enc, n_bins=15)

        print(f"ECE: {ece:.4f}")


        print("\nKey metrics:")
        display(summary)
        print("\nTop classes by support:")
        display(class_breakdown)

        raw_preds = model.predict(X_test).argmax(axis=1)
        cal_preds = calibrated_model.predict(X_test).argmax(axis=1)

        print("Accuracy identical:", np.all(raw_preds == cal_preds))
    
    return calibrated_model,scaler,label_encoder

In [7]:
nn_train(data)

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.9219 - loss: 0.4448 - top3_acc: 0.9908 - val_accuracy: 0.7339 - val_loss: 1.2165 - val_top3_acc: 0.8899 - learning_rate: 0.0010
Epoch 9/200
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - accuracy: 0.9514 - loss: 0.3671 - top3_acc: 0.9974 - val_accuracy: 0.7156 - val_loss: 1.2981 - val_top3_acc: 0.8792 - learning_rate: 0.0010
Epoch 10/200
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - accuracy: 0.9488 - loss: 0.3532 - top3_acc: 0.9954 - val_accuracy: 0.7370 - val_loss: 1.3448 - val_top3_acc: 0.8869 - learning_rate: 0.0010
Epoch 11/200
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step - accuracy: 0.9567 - loss: 0.3268 - top3_acc: 0.9967 - val_accuracy: 0.7370 - val_loss: 1.4176 - val_top3_acc: 0.8777 - learning_rate: 0.0010
Epoch 12/200
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.9678 - loss: 0.2967 - top3_acc: 1.0000 - val_accuracy: 0.7508 - val_loss: 1.4255 - val_top3_acc: 0.8792 - learning_rate: 0.0010
Epoch 13/200
24/24 ━━━━━━━━━━━━━━━━━━━━

/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/keras/src/backend/tensorflow/trainer.py:86: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


Saved model to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/pattern_classifier.keras
Saved scaler to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/scaler.joblib
Saved label encoder to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/label_encoder.joblib
Saved metadata to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/metadata.json


(<Functional name=functional_1, built=True>, StandardScaler(), LabelEncoder())

### Logistic Regression classifier

In [8]:
def lr_train(data=data):
    # Logistic regression stage intentionally skipped per latest workflow requirements.
    # The end-to-end classifier now relies solely on the neural network above.
    numeric_features = data.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_features:
        raise ValueError("No numeric features detected – please ensure embeddings/features are present.")

    X = data[numeric_features].fillna(0.0).values
    y = data[TARGET_COLUMN].astype(str).values
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    num_classes = len(label_encoder.classes_)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.25, random_state=42,stratify=y_encoded
    )

    from sklearn.linear_model import LogisticRegression

    logreg = LogisticRegression(max_iter=1000,n_jobs=-1)

    if EVALUATING_ENABLED:
        logreg.fit(X_train, y_train)
        y_pred = logreg.predict(X_test)
        report = classification_report(
            y_test,
            y_pred,
            target_names=label_encoder.classes_,
            output_dict=True,
            zero_division=0,
        )
        report_df = pd.DataFrame(report).T
        summary = report_df.loc[["accuracy", "macro avg", "weighted avg"]].round(3)
        class_breakdown = (
            report_df.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("support", ascending=False).head(8).round(3)
        )
        display(summary)
    else:
        logreg.fit(X, y)
        # Save model
        ARTIFACT_DIR = Path("../models/pattern_logreg_classifier").resolve()
        ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
        joblib.dump(logreg, ARTIFACT_DIR / "logistic_regression_model.joblib")
        joblib.dump(label_encoder, ARTIFACT_DIR / "label_encoder.joblib")
        joblib.dump(scaler, ARTIFACT_DIR / "scaler.joblib")
        print(f"Saved logistic regression model and artifacts to {ARTIFACT_DIR}")

    return logreg,scaler,label_encoder


In [9]:
lr_train(data)

Saved logistic regression model and artifacts to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_logreg_classifier


(LogisticRegression(max_iter=1000, n_jobs=-1),
 StandardScaler(),
 LabelEncoder())

### Ensemble training

In [10]:
from sklearn.model_selection import StratifiedKFold

raw_data = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/pattern_embeddings/gemini_pattern_embedding_v3.csv')
raw_data['pattern'].fillna('None', inplace=True)

synthetic_data       = raw_data[raw_data['file'].str.contains('pattern',na=False)]
verified_communities = raw_data[~raw_data['file'].str.contains('pattern',na=False)]
synthetic_data = synthetic_data.drop(columns=['file'])
verified_communities = verified_communities.drop(columns=['file'])
def get_folded_splits(fold_count=5,random_state=42,use_only_verified=True,min_samples_per_class=5):
    folded_data = []

    _sd = synthetic_data.copy()
    _vd = verified_communities.copy()

    _vd_c = _vd['pattern'].value_counts()
    _vp_i = _vd_c[_vd_c>=min_samples_per_class].index.tolist()
    _vd   = _vd[_vd['pattern'].isin(_vp_i)]

    skf = StratifiedKFold(
        n_splits=fold_count,
        shuffle=True,
        random_state=random_state
    )

    folds = list(skf.split(_vd, _vd['pattern']))

    for test_fold in range(fold_count):
        val_fold = (test_fold + 1) % fold_count
        train_folds = [
            i for i in range(fold_count)
            if i not in [test_fold, val_fold]
        ]

        train_idx = np.concatenate([folds[i][1] for i in train_folds])
        val_idx   = folds[val_fold][1]
        test_idx  = folds[test_fold][1]

        vd_train = _vd.iloc[train_idx]
        vd_val   = _vd.iloc[val_idx]
        vd_test  = _vd.iloc[test_idx]

        if use_only_verified:
            train_data = vd_train
        else:
            train_data = pd.concat([vd_train, _sd], ignore_index=True)

        folded_data.append(
            (train_data, vd_val, vd_test)
        )

    return folded_data

/tmp/ipykernel_291076/985136203.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  raw_data['pattern'].fillna('None', inplace=True)


In [11]:
from sklearn.model_selection import StratifiedKFold

In [12]:
folds = get_folded_splits(fold_count=5,use_only_verified=True,min_samples_per_class=12)
proba_dist = []

for fold in folds:
    train_data, test_data, val_data = fold
    nn_model,nn_scaler,nn_le = nn_train(train_data)
    nn_scaled_val = nn_scaler.transform(val_data.drop(columns=['pattern']))
    nn_scaled_test = nn_scaler.transform(test_data.drop(columns=['pattern']))
    nn_scaled_train = nn_scaler.transform(train_data.drop(columns=['pattern']))
    nn_proba_val = pd.DataFrame(nn_model.predict(nn_scaled_val),columns=nn_le.classes_)
    nn_proba_test = pd.DataFrame(nn_model.predict(nn_scaled_test),columns=nn_le.classes_)
    nn_proba_train = pd.DataFrame(nn_model.predict(nn_scaled_train),columns=nn_le.classes_)
    nn_proba_val['pattern'] = val_data['pattern'].values
    nn_proba_test['pattern'] = test_data['pattern'].values
    nn_proba_train['pattern'] = train_data['pattern'].values
    
    lr_model,lr_scaler,lr_le = lr_train(train_data)
    lr_scaled_val = lr_scaler.transform(val_data.drop(columns=['pattern']))
    lr_scaled_test = lr_scaler.transform(test_data.drop(columns=['pattern']))
    lr_scaled_train = lr_scaler.transform(train_data.drop(columns=['pattern']))
    lr_proba_val = pd.DataFrame(lr_model.predict_proba(lr_scaled_val),columns=lr_le.classes_)
    lr_proba_test = pd.DataFrame(lr_model.predict_proba(lr_scaled_test),columns=lr_le.classes_)
    lr_proba_train = pd.DataFrame(lr_model.predict_proba(lr_scaled_train),columns=lr_le.classes_)
    lr_proba_val['pattern'] = val_data['pattern'].values
    lr_proba_test['pattern'] = test_data['pattern'].values
    lr_proba_train['pattern'] = train_data['pattern'].values

    numeric_cols = nn_proba_val.select_dtypes(include=['number']).columns.tolist()
    meta_proba_val = pd.DataFrame()
    meta_proba_test = pd.DataFrame()
    meta_proba_train = pd.DataFrame()
    meta_proba_val = (nn_proba_val[numeric_cols] + lr_proba_val[numeric_cols]*0.7)/2
    meta_proba_test = (nn_proba_test[numeric_cols] + lr_proba_test[numeric_cols]*0.7)/2
    meta_proba_train = (nn_proba_train[numeric_cols] + lr_proba_train[numeric_cols]*0.7)/2
    meta_proba_val['pattern'] = val_data['pattern'].values
    meta_proba_test['pattern'] = test_data['pattern'].values
    meta_proba_train['pattern'] = train_data['pattern'].values
    proba_dist.append({
        'train':train_data,
        'val':val_data,
        'test':test_data,
        'nn_proba_val':nn_proba_val,
        'lr_proba_val':lr_proba_val,
        'meta_proba_val':meta_proba_val,
        'nn_proba_test':nn_proba_test,
        'lr_proba_test':lr_proba_test,
        'meta_proba_test':meta_proba_test,
        'nn_proba_train':nn_proba_train,
        'lr_proba_train':lr_proba_train,
        'meta_proba_train':meta_proba_train,
    })

Epoch 1/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 6s 713ms/step - accuracy: 0.0811 - loss: 3.0742 - top3_acc: 0.3243 - val_accuracy: 0.1042 - val_loss: 2.4016 - val_top3_acc: 0.3542 - learning_rate: 0.0010
Epoch 2/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 222ms/step - accuracy: 0.2793 - loss: 2.3261 - top3_acc: 0.5586 - val_accuracy: 0.1458 - val_loss: 2.3162 - val_top3_acc: 0.4375 - learning_rate: 0.0010
Epoch 3/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step - accuracy: 0.4775 - loss: 1.6204 - top3_acc: 0.8198 - val_accuracy: 0.2500 - val_loss: 2.2189 - val_top3_acc: 0.5208 - learning_rate: 0.0010
Epoch 4/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step - accuracy: 0.7207 - loss: 1.1575 - top3_acc: 0.9279 - val_accuracy: 0.3750 - val_loss: 2.1232 - val_top3_acc: 0.6042 - learning_rate: 0.0010
Epoch 5/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step - accuracy: 0.6937 - loss: 1.0372 - top3_acc: 0.9189 - val_accuracy: 0.3750 - val_loss: 2.0497 - val_top3_acc: 0.6458 - learning_rate: 0.0010
Epoch 6/200
2/2 ━━━━━━━━━━━━━━━━━━━

/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/keras/src/backend/tensorflow/trainer.py:86: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


Saved model to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/pattern_classifier.keras
Saved scaler to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/scaler.joblib
Saved label encoder to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/label_encoder.joblib
Saved metadata to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/metadata.json


/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
Saved logistic regression model and artifacts to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_logreg_classifier


...

/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/keras/src/backend/tensorflow/trainer.py:86: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


Saved model to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/pattern_classifier.keras
Saved scaler to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/scaler.joblib
Saved label encoder to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/label_encoder.joblib
Saved metadata to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/metadata.json


/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 223ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
Saved logistic regression model and artifacts to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_logreg_classifier


/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [13]:
dist_test = pd.DataFrame(columns=proba_dist[0]['meta_proba_test'].columns)
dist_val = pd.DataFrame(columns=proba_dist[0]['meta_proba_val'].columns)
for fold_proba in proba_dist:
    dist_test = pd.concat([dist_test,fold_proba['meta_proba_test']],ignore_index=True)
    dist_val = pd.concat([dist_val,fold_proba['meta_proba_val']],ignore_index=True)

/tmp/ipykernel_291076/2107439126.py:4: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dist_test = pd.concat([dist_test,fold_proba['meta_proba_test']],ignore_index=True)
/tmp/ipykernel_291076/2107439126.py:5: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dist_val = pd.concat([dist_val,fold_proba['meta_proba_val']],ignore_index=True)


In [14]:
def get_prediction(row, class_threshold=0.5,none_threshold=0.5):
    high_prob = row.max()
    if high_prob >= class_threshold:
        max_class = row.idxmax()
        return max_class
    elif high_prob <= none_threshold:
        return 'None Type'
    return "Other"


def classify(class_threshold,none_threshold,data):
    y_true = data['pattern'].fillna('None Type')
    y_pred = data.drop('pattern', axis=1).apply(get_prediction, class_threshold=class_threshold,none_threshold=none_threshold, axis=1)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    macro = report['macro avg']
    report_df = pd.DataFrame(report).T
    return macro['precision'], macro['recall'], macro['f1-score'],report_df

In [15]:
best = None
best_class_wise = pd.DataFrame()
for th_t in range(0,100,10):
    for thn in range(0,100,100):
        th = th_t/100
        thn=thn/100
        precision, recall, f1, report_df = classify(th,thn,dist_val)
        
        best_class_wise[str(th)+':'+str(thn)] = report_df['f1-score']

        if best is None or f1 > best[0]:
            best = (f1, th, precision, recall)
        print(f"Threshold: {th} => Precision: {precision}, Recall: {recall}, F1-Score: {f1}")

print("-"*10)
print(f"Best Threshold: {best[1]} => Precision: {best[2]}, Recall: {best[3]}, F1-Score: {best[0]}")
best_class_wise['best_threshold'] = best_class_wise.idxmax(axis=1)

Threshold: 0.0 => Precision: 0.40361325086649125, Recall: 0.37481654913688006, F1-Score: 0.3793805852064087
Threshold: 0.1 => Precision: 0.40361325086649125, Recall: 0.37481654913688006, F1-Score: 0.3793805852064087
Threshold: 0.2 => Precision: 0.37864385399892647, Recall: 0.33733489422319207, F1-Score: 0.3468909073129797
Threshold: 0.3 => Precision: 0.4320762305495262, Recall: 0.3079543646365455, F1-Score: 0.3434286390192215
Threshold: 0.4 => Precision: 0.40168542242071653, Recall: 0.23222476745083132, F1-Score: 0.2836930564562144
Threshold: 0.5 => Precision: 0.4353766420361248, Recall: 0.12940050911460485, F1-Score: 0.18290154000076417
Threshold: 0.6 => Precision: 0.5016666666666667, Recall: 0.07596835616117532, F1-Score: 0.12184146504661482
Threshold: 0.7 => Precision: 0.5518181818181819, Recall: 0.058358904442681034, F1-Score: 0.1023015133359961
Threshold: 0.8 => Precision: 0.25, Recall: 0.014774182427107957, F1-Score: 0.027731092436974792
Threshold: 0.9 => Precision: 0.0, Recall: 

In [16]:
best_class_wise

,0.0:0.0,0.1:0.0,0.2:0.0,0.3:0.0,0.4:0.0,0.5:0.0,0.6:0.0,0.7:0.0,0.8:0.0,0.9:0.0,best_threshold
Classical Models,0.648148,0.648148,0.654206,0.660000,0.681319,0.582278,0.411765,0.310345,0.117647,0.0,0.4:0.0
LLM Results Evaluation,0.105263,0.105263,0.111111,0.125000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.3:0.0
LLM based Multimodal Generative Prompting,0.416667,0.416667,0.434783,0.444444,0.450000,0.363636,0.266667,0.206897,0.000000,0.0,0.4:0.0
Model Abstraction Pattern,0.250000,0.250000,0.250000,0.285714,0.105263,0.000000,0.000000,0.000000,0.000000,0.0,0.3:0.0
Modular LLM Agent Architectures,0.400000,0.400000,0.411765,0.387097,0.370370,0.095238,0.095238,0.095238,0.000000,0.0,0.2:0.0
None,0.441379,0.441379,0.441379,0.436090,0.392857,0.365591,0.227848,0.189189,0.088235,0.0,0.0:0.0
Preprocessing Text and Numerical Data,0.385965,0.385965,0.392857,0.377358,0.291667,0.205128,0.057143,0.057143,0.000000,0.0,0.2:0.0
Retrieval Augmented Generation(RAG) Optimization for LLMs,0.465116,0.465116,0.465116,0.473684,0.363636,0.160000,0.095238,0.095238,0.000000,0.0,0.3:0.0
Tool Use for LLMs,0.301887,0.301887,0.307692,0.244898,0.181818,0.057143,0.064516,0.068966,0.071429,0.0,0.2:0.0
accuracy,0.439850,0.439850,0.439850,0.402256,0.319549,0.203008,0.116541,0.086466,0.026316,0.0,0.0:0.0


In [17]:
best_df = pd.DataFrame()
best_df['Class Threshold'] = best_class_wise['best_threshold'].apply(lambda x: x.split(":")[0])
best_df['None Threshold'] = best_class_wise['best_threshold'].apply(lambda x: x.split(":")[1])
best_df

,Class Threshold,None Threshold
Classical Models,0.4,0.0
LLM Results Evaluation,0.3,0.0
LLM based Multimodal Generative Prompting,0.4,0.0
Model Abstraction Pattern,0.3,0.0
Modular LLM Agent Architectures,0.2,0.0
None,0.0,0.0
Preprocessing Text and Numerical Data,0.2,0.0
Retrieval Augmented Generation(RAG) Optimization for LLMs,0.3,0.0
Tool Use for LLMs,0.2,0.0
accuracy,0.0,0.0


In [31]:
def get_prediction_trained(row):
    high_prob = row.max()
    max_class = row.idxmax()
    if float(best_df['Class Threshold'][max_class]) <= float(high_prob):
        return max_class
    return "Other"


def classify_trained(data):
    print(data.shape)
    y_true = data['pattern'].fillna('None Type')
    y_pred = data.drop('pattern', axis=1).apply(get_prediction_trained, axis=1)
    print(f"pricision:{precision} recall:{recall} f1:{f1}")
    display(y_pred.value_counts())

    ignore_class = "Other"
    labels = [c for c in y_true.unique() if c != ignore_class]
    labels.extend([c for c in y_pred.unique() if c != ignore_class])
    labels = list(set(labels))
    report = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
    macro = report['macro avg']
    report_df = pd.DataFrame(report).T
    return macro['precision'], macro['recall'], macro['f1-score'],report_df

precision, recall, f1, report_df = classify_trained(dist_val)
print(f"Precision: {precision}, Recall: {recall}, F1-Score: {f1}")
with_threshold_report = report_df.T
report_df

(266, 10)
pricision:0.49711761429957896 recall:0.35553122453240654 f1:0.39660118270399036


None                                                         81
Classical Models                                             44
Other                                                        39
Tool Use for LLMs                                            25
Preprocessing Text and Numerical Data                        23
Retrieval Augmented Generation(RAG) Optimization for LLMs    18
LLM based Multimodal Generative Prompting                    14
Modular LLM Agent Architectures                              14
Model Abstraction Pattern                                     5
LLM Results Evaluation                                        3
Name: count, dtype: int64

Precision: 0.49711761429957885, Recall: 0.35553122453240654, F1-Score: 0.3966011827039904


,precision,recall,f1-score,support
Modular LLM Agent Architectures,0.500000,0.350000,0.411765,20.0
LLM based Multimodal Generative Prompting,0.642857,0.346154,0.450000,26.0
None,0.395062,0.500000,0.441379,64.0
Retrieval Augmented Generation(RAG) Optimization for LLMs,0.500000,0.450000,0.473684,20.0
Preprocessing Text and Numerical Data,0.478261,0.333333,0.392857,33.0
LLM Results Evaluation,0.333333,0.076923,0.125000,13.0
Model Abstraction Pattern,0.600000,0.187500,0.285714,16.0
Tool Use for LLMs,0.320000,0.296296,0.307692,27.0
Classical Models,0.704545,0.659574,0.681319,47.0
micro avg,0.488987,0.417293,0.450304,266.0


In [27]:
def get_prediction_no_threshold(row):
    max_class = row.idxmax()
    return max_class


def classify_trained_no_threshold(data):
    print(data.shape)
    y_true = data['pattern'].fillna('None Type')
    y_pred = data.drop('pattern', axis=1).apply(get_prediction_no_threshold, axis=1)
    print(y_true)
    print(y_pred)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    macro = report['macro avg']
    report_df = pd.DataFrame(report).T
    return macro['precision'], macro['recall'], macro['f1-score'],report_df

precision, recall, f1, report_df = classify_trained_no_threshold(dist_val)
print(f"Precision: {precision}, Recall: {recall}, F1-Score: {f1}")
without_threshold_report = report_df.T
report_df

(266, 10)
0       Classical Models
1       Classical Models
2       Classical Models
3       Classical Models
4       Classical Models
             ...        
261    Tool Use for LLMs
262    Tool Use for LLMs
263    Tool Use for LLMs
264    Tool Use for LLMs
265    Tool Use for LLMs
Name: pattern, Length: 266, dtype: object
0                           Classical Models
1                           Classical Models
2                           Classical Models
3                                       None
4      Preprocessing Text and Numerical Data
                       ...                  
261                        Tool Use for LLMs
262          Modular LLM Agent Architectures
263                                     None
264                        Tool Use for LLMs
265                        Tool Use for LLMs
Length: 266, dtype: object
Precision: 0.40361325086649125, Recall: 0.37481654913688006, F1-Score: 0.3793805852064087


,precision,recall,f1-score,support
Classical Models,0.573770,0.744681,0.648148,47.00000
LLM Results Evaluation,0.166667,0.076923,0.105263,13.00000
LLM based Multimodal Generative Prompting,0.454545,0.384615,0.416667,26.00000
Model Abstraction Pattern,0.375000,0.187500,0.250000,16.00000
Modular LLM Agent Architectures,0.466667,0.350000,0.400000,20.00000
None,0.395062,0.500000,0.441379,64.00000
Preprocessing Text and Numerical Data,0.458333,0.333333,0.385965,33.00000
Retrieval Augmented Generation(RAG) Optimization for LLMs,0.434783,0.500000,0.465116,20.00000
Tool Use for LLMs,0.307692,0.296296,0.301887,27.00000
accuracy,0.439850,0.439850,0.439850,0.43985


In [20]:
with_threshold_report.rename(index={
    'precision':'precision (with thresholds)',
    'recall':'recall (with thresholds)',
    'f1-score':'f1-score (with thresholds)'
},inplace=True)

without_threshold_report.rename(index={
    'precision':'precision (without thresholds)',
    'recall':'recall (without thresholds)',
    'f1-score':'f1-score (without thresholds)'
},inplace=True)

final_report = pd.concat([with_threshold_report,without_threshold_report])

In [21]:
final_report.to_csv('result/scores/threshold_report.csv')

In [22]:
final_report

,Classical Models,LLM Results Evaluation,LLM based Multimodal Generative Prompting,Model Abstraction Pattern,Modular LLM Agent Architectures,None,Preprocessing Text and Numerical Data,Retrieval Augmented Generation(RAG) Optimization for LLMs,Tool Use for LLMs,micro avg,macro avg,weighted avg,accuracy
precision (with thresholds),0.704545,0.333333,0.642857,0.600000,0.500000,0.395062,0.478261,0.500000,0.320000,0.488987,0.497118,0.501759,NaN
recall (with thresholds),0.659574,0.076923,0.346154,0.187500,0.350000,0.500000,0.333333,0.450000,0.296296,0.417293,0.355531,0.417293,NaN
f1-score (with thresholds),0.681319,0.125000,0.450000,0.285714,0.411765,0.441379,0.392857,0.473684,0.307692,0.450304,0.396601,0.440405,NaN
support,47.000000,13.000000,26.000000,16.000000,20.000000,64.000000,33.000000,20.000000,27.000000,532.000000,532.000000,532.000000,NaN
precision (without thresholds),0.573770,0.166667,0.454545,0.375000,0.466667,0.395062,0.458333,0.434783,0.307692,NaN,0.403613,0.427435,0.43985
recall (without thresholds),0.744681,0.076923,0.384615,0.187500,0.350000,0.500000,0.333333,0.500000,0.296296,NaN,0.374817,0.439850,0.43985
f1-score (without thresholds),0.648148,0.105263,0.416667,0.250000,0.400000,0.441379,0.385965,0.465116,0.301887,NaN,0.379381,0.425200,0.43985
support,47.000000,13.000000,26.000000,16.000000,20.000000,64.000000,33.000000,20.000000,27.000000,NaN,266.000000,266.000000,0.43985


### Second level Model

In [23]:
from sklearn.linear_model import LogisticRegression

lr_pred = pd.DataFrame(columns=['prediction','pattern'])
c=0
for fold in proba_dist:
    print("Processing fold:",c)

    # train set
    X_meta_train = fold['meta_proba_train'].drop(columns=['pattern'])
    X_lr_train = fold['lr_proba_train'].drop(columns=['pattern'])
    X_nn_train = fold['nn_proba_train'].drop(columns=['pattern'])
    y_temp_train = fold['meta_proba_train']['pattern']

    X_meta_train = X_meta_train.add_prefix('meta_')
    X_lr_train = X_lr_train.add_prefix('lr_')
    X_nn_train = X_nn_train.add_prefix('nn_')

    # Validation set
    X_meta_val = fold['meta_proba_val'].drop(columns=['pattern'])
    X_lr_val = fold['lr_proba_val'].drop(columns=['pattern'])
    X_nn_val = fold['nn_proba_val'].drop(columns=['pattern'])
    y_temp_val = fold['meta_proba_val']['pattern']

    X_meta_val = X_meta_val.add_prefix('meta_')
    X_lr_val = X_lr_val.add_prefix('lr_')
    X_nn_val = X_nn_val.add_prefix('nn_')

    X_meta_val = pd.concat([X_meta_train,X_meta_val], axis=0)
    X_lr_val = pd.concat([X_lr_train,X_lr_val], axis=0)
    X_nn_val = pd.concat([X_nn_train,X_nn_val], axis=0)
    y_temp_val = pd.concat([y_temp_train,y_temp_val], axis=0)
    
    # Test set
    X_meta_test = fold['meta_proba_test'].drop(columns=['pattern'])
    X_lr_test = fold['lr_proba_test'].drop(columns=['pattern'])
    X_nn_test = fold['nn_proba_test'].drop(columns=['pattern'])
    y_temp_test = fold['meta_proba_test']['pattern']

    X_meta_test = X_meta_test.add_prefix('meta_')
    X_lr_test = X_lr_test.add_prefix('lr_')
    X_nn_test = X_nn_test.add_prefix('nn_')


    dataset_val = pd.concat([X_meta_val,X_nn_val,X_lr_val,y_temp_val], axis=1)
    dataset_test = pd.concat([X_meta_test,X_nn_test,X_lr_test,y_temp_test], axis=1)

    # display(dataset)

    lr_sec_model,le_sec_scaler,le_sec_encoder = lr_train(dataset_val)
    lr_sec_scaled = le_sec_scaler.transform(dataset_test.drop(columns=['pattern']))
    lr_sec_pred = pd.DataFrame(columns=['prediction','pattern'])
    lr_sec_pred['prediction'] = lr_sec_model.predict(lr_sec_scaled)
    lr_sec_pred['pattern'] = dataset_test['pattern'].values
    lr_pred = pd.concat([lr_pred,lr_sec_pred])
    c+=1

Processing fold: 0
Saved logistic regression model and artifacts to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_logreg_classifier
Processing fold: 1


/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


Saved logistic regression model and artifacts to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_logreg_classifier
Processing fold: 2
Saved logistic regression model and artifacts to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_logreg_classifier
Processing fold: 3
Saved logistic regression model and artifacts to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_logreg_classifier
Processing fold: 4
Saved logistic regression model and artifacts to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_logreg_classifier


/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [24]:
report = classification_report(lr_pred['prediction'], lr_pred['pattern'], output_dict=True, zero_division=0)
pd.DataFrame(report).T

,precision,recall,f1-score,support
Classical Models,0.744681,0.648148,0.693069,54.000000
LLM Results Evaluation,0.230769,0.428571,0.300000,7.000000
LLM based Multimodal Generative Prompting,0.307692,0.470588,0.372093,17.000000
Model Abstraction Pattern,0.250000,0.333333,0.285714,12.000000
Modular LLM Agent Architectures,0.300000,0.333333,0.315789,18.000000
None,0.546875,0.443038,0.489510,79.000000
Preprocessing Text and Numerical Data,0.454545,0.441176,0.447761,34.000000
Retrieval Augmented Generation(RAG) Optimization for LLMs,0.450000,0.500000,0.473684,18.000000
Tool Use for LLMs,0.259259,0.259259,0.259259,27.000000
accuracy,0.458647,0.458647,0.458647,0.458647
